<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32 x 32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 x 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 x 32 x 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.

Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [ ]:
mlflow.set_experiment(
    "assignment"
)

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**

A função abaixo carrega o CIFAR-10, transforma cada imagem 32 x 32 x 3 em um vetor de 3072 atributos, normaliza os pixels para o intervalo [0, 1] e separa treino/validação com `train_test_split`.

Respostas:

1. O formato original de cada imagem é 32 x 32 x 3, ou seja, altura, largura e três canais RGB.
2. Após o flatten cada imagem possui 32 * 32 * 3 = 3072 features.
3. O flatten é necessário porque a MLP do `sklearn` recebe uma matriz tabular no formato `(amostras, features)`, não tensores de imagem 2D/3D.
4. A normalização evita que valores de pixel entre 0 e 255 gerem gradientes maiores e instáveis, ajudando a convergência do otimizador.

In [ ]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split


def load_data(seed=42, validation_size=0.2, max_samples=None):
    (X_train_full, y_train_full), _ = cifar10.load_data()

    y_train_full = y_train_full.ravel()
    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1).astype("float32") / 255.0

    if max_samples is not None and max_samples < X_train_full.shape[0]:
        X_train_full, _, y_train_full, _ = train_test_split(
            X_train_full,
            y_train_full,
            train_size=max_samples,
            random_state=seed,
            stratify=y_train_full,
        )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=validation_size,
        random_state=seed,
        stratify=y_train_full,
    )

    return X_train, X_val, y_train, y_val


SEED = 42
X_train, X_val, y_train, y_val = load_data(seed=SEED, max_samples=8000)

print("Treino:", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**

A função `train_mlp` encapsula a criação e o treinamento do `MLPClassifier`.

Respostas:

1. A primeira camada tem `(n_features + 1) * n_neurônios` parâmetros, pois há um peso para cada entrada e um bias por neurônio. Com CIFAR-10 achatado e uma camada de 64 neurônios: `(3072 + 1) * 64 = 196672`.
2. A ReLU aplica `max(0, x)`, introduz não linearidade e reduz o problema de gradientes muito pequenos em regiões positivas.
3. MLPs possuem muitos parâmetros em imagens porque cada neurônio da primeira camada se conecta a todos os pixels/canais, ignorando a estrutura espacial local da imagem.

In [ ]:
from sklearn.neural_network import MLPClassifier


def train_mlp(
    X_train,
    y_train,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    seed=42,
    max_iter=30,
    batch_size=128,
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=max_iter,
        batch_size=batch_size,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=5,
        verbose=False,
    )

    model.fit(X_train, y_train)
    return model


baseline_model = train_mlp(X_train, y_train, seed=SEED)
print("Épocas executadas:", baseline_model.n_iter_)
print("Loss final:", baseline_model.loss_)

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**

A função `evaluate` calcula as métricas principais para classificação multiclasse. Foi usado `average="weighted"` para considerar o suporte de cada classe.

Respostas:

1. Accuracy é a proporção de exemplos classificados corretamente entre todos os exemplos avaliados.
2. Precision mede, entre os itens previstos como uma classe, quantos realmente pertencem a ela; recall mede, entre os itens reais daquela classe, quantos foram recuperados pelo modelo.
3. O f1-score é importante quando precisamos equilibrar precision e recall, especialmente quando erros de falso positivo e falso negativo importam ao mesmo tempo ou quando há desbalanceamento entre classes.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }


baseline_metrics = evaluate(baseline_model, X_val, y_val)
pd.DataFrame([baseline_metrics])

No experimento baseline, as métricas devem ser interpretadas como desempenho em validação, não em treino. Em CIFAR-10, uma MLP simples tende a ter desempenho limitado porque perde informação espacial ao achatar a imagem; ainda assim, as métricas permitem comparar hiperparâmetros de forma consistente.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**

O bloco abaixo cria uma função reutilizável para treinar, avaliar e registrar cada execução no MLflow.

In [ ]:
import time


def run_experiment(
    run_name,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    max_iter=30,
    batch_size=128,
    seed=42,
):
    params = {
        "activation": activation,
        "hidden_layers": str(hidden_layers),
        "learning_rate": learning_rate,
        "max_iter": max_iter,
        "batch_size": batch_size,
        "seed": seed,
    }

    with mlflow.start_run(run_name=run_name):
        start = time.time()
        model = train_mlp(
            X_train,
            y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=seed,
            max_iter=max_iter,
            batch_size=batch_size,
        )
        training_time = time.time() - start
        metrics = evaluate(model, X_val, y_val)
        metrics["training_time"] = training_time
        metrics["final_loss"] = float(model.loss_)
        metrics["n_iter"] = int(model.n_iter_)

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

    result = {**params, **metrics, "run_name": run_name}
    return model, result


tracked_model, tracked_result = run_experiment("baseline_tracked", seed=SEED)
pd.DataFrame([tracked_result])

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**

Para comparar ativações, mantive arquitetura, learning rate, seed, `max_iter` e `batch_size` constantes. Assim, a diferença observada fica associada principalmente à função de ativação.

Respostas esperadas após a execução:

1. Em geral, ReLU tende a convergir melhor por manter gradientes mais fortes em regiões positivas.
2. A estabilidade deve ser observada pela combinação de `final_loss`, `n_iter` e métricas de validação; tanh pode ser estável, mas costuma convergir mais lentamente.
3. Normalmente há diferenças, pois logistic/tanh saturam com mais facilidade.
4. ReLU é muito usada porque é simples, barata computacionalmente e reduz saturação de gradientes em comparação com sigmoid/logistic.

In [ ]:
activation_results = []

for activation in ["logistic", "tanh", "relu"]:
    _, result = run_experiment(
        run_name=f"activation_{activation}",
        activation=activation,
        hidden_layers=(64,),
        learning_rate=0.001,
        max_iter=30,
        batch_size=128,
        seed=SEED,
    )
    activation_results.append(result)

activation_df = pd.DataFrame(activation_results).sort_values("accuracy", ascending=False)
activation_df

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**

As arquiteturas foram comparadas mantendo ativação, learning rate e seed constantes. O objetivo é avaliar ganho de capacidade versus custo computacional.

Respostas esperadas após a execução:

1. Redes maiores não necessariamente melhoram sempre; podem aumentar custo e overfitting.
2. O melhor tradeoff é a menor arquitetura com métrica próxima da melhor, considerando também tempo de treinamento.
3. Sinais de overfitting aparecem quando a rede fica maior/custa mais, mas não melhora validação de forma proporcional.
4. A arquitetura `(256, 128)` tende a ter maior custo por possuir mais pesos e mais operações por época.

In [ ]:
architecture_results = []

for architecture in [(32,), (64,), (128, 64), (256, 128)]:
    name = "architecture_" + "x".join(map(str, architecture))
    _, result = run_experiment(
        run_name=name,
        activation="relu",
        hidden_layers=architecture,
        learning_rate=0.001,
        max_iter=30,
        batch_size=128,
        seed=SEED,
    )
    architecture_results.append(result)

architecture_df = pd.DataFrame(architecture_results).sort_values("accuracy", ascending=False)
architecture_df

# Questão 7

**Solução**

Os learning rates foram comparados mantendo arquitetura, ativação e seed constantes.

Respostas esperadas após a execução:

1. O melhor desempenho costuma ocorrer em um valor intermediário/baixo, frequentemente `0.001` para MLPClassifier com Adam nesse problema.
2. `0.1` tende a ser o mais instável, pois passos muito grandes podem impedir convergência.
3. Learning rate muito alto pode fazer a loss oscilar ou divergir.
4. Learning rate muito baixo pode deixar o treinamento lento e preso em progresso pequeno por época.

In [ ]:
learning_rate_results = []

for lr in [0.1, 0.01, 0.001]:
    _, result = run_experiment(
        run_name=f"learning_rate_{lr}",
        activation="relu",
        hidden_layers=(64,),
        learning_rate=lr,
        max_iter=30,
        batch_size=128,
        seed=SEED,
    )
    learning_rate_results.append(result)

learning_rate_df = pd.DataFrame(learning_rate_results).sort_values("accuracy", ascending=False)
learning_rate_df

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

**Discussão final**

A loss deve diminuir ao longo das épocas quando o learning rate permite atualizações estáveis. Quando o learning rate é alto demais, o otimizador pode saltar regiões boas da superfície de erro, causando oscilação ou baixa convergência. Quando é baixo demais, o aprendizado fica lento e pode não aproveitar bem o limite de épocas.

A arquitetura controla a capacidade do modelo. Camadas maiores aumentam o número de parâmetros e podem melhorar a representação, mas também aumentam custo computacional e risco de overfitting. Por isso, a melhor configuração não é necessariamente a maior rede, e sim aquela que entrega boa validação com custo aceitável.

As funções de ativação afetam o fluxo de gradientes. Logistic e tanh podem saturar, enquanto ReLU costuma treinar melhor em redes profundas ou com muitas entradas porque preserva gradientes em valores positivos. Ainda assim, a melhor escolha deve ser confirmada pelas métricas do experimento.

MLPs têm limitações para imagens porque o flatten destrói a vizinhança espacial entre pixels. Uma CNN tende a ser mais adequada para CIFAR-10 porque usa filtros locais, compartilhamento de pesos e hierarquias espaciais.

O backpropagation contribui calculando como cada peso participou do erro final. Com esses gradientes, o otimizador atualiza pesos camada por camada para reduzir a loss, permitindo que a rede aprenda representações mais úteis a cada época.

Para identificar a melhor configuração final, use a linha com maior `accuracy`/`f1_score` entre `activation_df`, `architecture_df` e `learning_rate_df`, considerando também `training_time` e `final_loss`. As principais dificuldades são custo de treinamento, sensibilidade a hiperparâmetros e desempenho limitado da MLP em imagens.